# 1 — Coding Essentials

This is a workbook, not a lecture. Read a little, run a little, write a little.

**How it works**

* Every section ends with **Your turn** — a cell with gaps for you to fill in.
* Those cells finish with `check('1.3', ...)`. Run it and you get a ✅, or a ❌
  that tells you exactly what is wrong.
* Stuck? Run `hint('1.3')` in any cell.
* `todo()` is a placeholder. Every single one has to be replaced by your code.

**Two habits worth forming today**

1. Run cells **in order, top to bottom.** A notebook remembers everything you
   have run, so a cell can fail simply because you skipped the one above it.
   When things get confusing: *Kernel → Restart and Run All*.
2. When something breaks, **read the last line of the error first.** That is
   where Python says what it could not do.

`Shift + Enter` runs a cell and moves on. `Ctrl + Enter` runs it and stays put.

---

**What is in here:** variables, text, collections, branching, loops,
functions, errors, NumPy, pandas, classes, imports.

**What is deliberately not in here:** plotting (notebook 2), regression
(notebook 3), simulation (notebook 4). One skill per notebook.

In [ ]:
# Run me first. This makes workbook.py importable whatever folder Jupyter
# started in, then pulls in the three helpers you will use all the way through.
import sys
from pathlib import Path

for candidate in [Path.cwd(), Path.cwd() / 'Learn_To_Code', *Path.cwd().parents]:
    if (candidate / 'workbook.py').exists():
        sys.path.insert(0, str(candidate))
        break

from workbook import check, hint, todo, ensure_data, DATA_DIR

ensure_data()   # builds the workbook CSVs on your Desktop the first time only

print('Ready. Data lives in:', DATA_DIR)

That cell did four things worth understanding, because you will write this
pattern yourself one day:

* `from pathlib import Path` — the modern way to handle file locations. `Path`
  objects work the same on macOS and Windows, which plain strings do not.
* the `for` loop hunts upwards from wherever Jupyter started until it finds
  `workbook.py`, and adds that folder to `sys.path` — the list of places Python
  looks when you `import` something.
* `from workbook import ...` pulls the helper functions out of that file.
* `ensure_data()` builds the two CSV files these notebooks read. They are written
  to a folder on your **Desktop**, never into the project — a repository is for
  code, and generated data has no business in one. The first run makes them; every
  run after that finds them already there.

If it printed a path, you are ready. If it raised `ModuleNotFoundError`, the
notebook has been moved away from `workbook.py` — put it back.

## 1. Variables and types

A variable is a **name pointing at a value**. The `=` sign means "make this name
point at that value". It is not the `=` of mathematics — `x = x + 1` is perfectly
sensible code and nonsense algebra.

Python works out the type for you, but the type is real and it matters. `250` and
`250.0` are different things, and `'250'` is not a number at all.

In [ ]:
ticker  = 'ACME'     # str   — text, always in quotes
shares  = 250        # int   — whole number
price   = 142.50     # float — number with a decimal point
is_open = True       # bool  — True or False, capitalised, no quotes

print(ticker,  type(ticker))
print(shares,  type(shares))
print(price,   type(price))
print(is_open, type(is_open))

In [ ]:
# The name points at a value. Point it somewhere else and the old value is gone.
shares = shares + 50
print('after buying 50 more:', shares)

# Types decide what an operator means. Same symbol, very different result:
print(3 * 4)          # 12          — arithmetic
print('3' * 4)        # 3333        — repeating text
print(7 / 2)          # 3.5         — / always gives a float
print(7 // 2, 7 % 2)  # 3 1         — floor division and remainder

**Naming.** Use `lower_case_with_underscores`, and say what the thing *is*:
`shares_outstanding` beats `s`. You will read your code far more often than you
write it.

**One trap:** do not use names Python already has — `list`, `sum`, `type`, `str`,
`id`, `input`. Assigning to them quietly breaks the built-in of the same name,
and the error shows up several cells later somewhere that looks unrelated.

### One thing about floats, now rather than later

Computers store decimals in binary, and `0.1` has no exact binary form — the same
way `1/3` has no exact decimal form. So the arithmetic is very slightly off, and
it always will be.

In [ ]:
import math

print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)                      # False. Not a bug.
print(math.isclose(0.1 + 0.2, 0.3))          # this is how you ask the question

Two habits follow:

* **never compare two floats with `==`** — ask whether the gap is small enough;
* **do not use floats for money you have to reconcile to the cent.**

It is also why `check()` in this workbook accepts an answer that is close enough
rather than demanding the exact bits.

### Your turn

In [ ]:
# Create four variables with exactly these values:
#   ticker  : the text ACME
#   shares  : the whole number 250
#   price   : the number 142.5, written with a decimal point
#   is_open : true, spelled the way Python spells it

ticker  = todo()
shares  = todo()
price   = todo()
is_open = todo()

check('1.1', ticker, shares, price, is_open)

## 2. Printing, and f-strings

`print()` is how you look at things. An **f-string** is how you print things
*well*: put `f` in front of a string and anything inside `{curly braces}` is
evaluated and dropped in.

In [ ]:
ticker, shares, price = 'ACME', 250, 142.50
value = shares * price

print('plain:', ticker, value)                 # fine for a quick look
print(ticker + ' is worth ' + str(value))      # works, but clumsy and fragile
print(f'{ticker} is worth {value}')            # f-string: readable
print(f'{ticker} is worth ${value:,.2f}')      # ...and formatted

The part after the colon is a **format spec**. These five cover almost everything
you will need:

| Spec       | Example                  | Result       | Use it for |
| ---------- | ------------------------ | ------------ | ---------- |
| `.2f`      | `f'{1234.5678:.2f}'`     | `1234.57`    | prices |
| `,.2f`     | `f'{1234.5678:,.2f}'`    | `1,234.57`   | money with a thousands separator |
| `.1%`      | `f'{0.0734:.1%}'`        | `7.3%`       | returns — it multiplies by 100 for you |
| `>10.2f`   | `f'{1234.5678:>10.2f}'`  | `···1234.57` | lining up a column of numbers |
| `.3e`      | `f'{0.000123:.3e}'`      | `1.230e-04`  | very large or very small numbers |

Formatting changes how a number is *displayed*, never the number itself.

In [ ]:
for name, ret in [('ACME', 0.0734), ('BOLT', -0.0121), ('CRUX', 0.0009)]:
    print(f'{name:<6}{ret:>8.2%}')

# A debugging trick worth knowing: = inside the braces prints the expression too.
print(f'{value = }')

### Your turn

In [ ]:
# Turn position_value into a string that reads exactly:  35,625.00
# (thousands separator, two decimal places, no dollar sign)
position_value = 250 * 142.5

text = todo()

print(text)
check('1.2', text)

## 3. Lists

A list is an ordered, changeable sequence. Square brackets, commas between items.

**Indexing starts at 0.** The first item is `prices[0]`. Negative numbers count
back from the end, so `prices[-1]` is the last one — you will use that constantly.

In [ ]:
prices = [101.2, 99.8, 103.4, 100.0, 105.5, 108.1]

print('length      ', len(prices))
print('first       ', prices[0])
print('last        ', prices[-1])
print('second last ', prices[-2])

**Slicing** takes a range: `prices[start:stop]`. The `start` is included, the
`stop` is **not**. That off-by-one feels wrong for about a week and then becomes
second nature — its great virtue is that `prices[:3]` and `prices[3:]` split the
list perfectly in two with no overlap and nothing lost.

In [ ]:
print(prices[0:3])   # items 0, 1, 2
print(prices[:3])    # same thing — a missing start means "from the beginning"
print(prices[3:])    # from item 3 to the end
print(prices[-2:])   # the last two
print(prices[::2])   # every second item

In [ ]:
# Lists are mutable: you can change them in place.
prices.append(110.0)      # add to the end
print(prices)

prices[0] = 102.0         # overwrite one item
print(prices)

removed = prices.pop()    # take the last one off, and hand it back
print('removed', removed, '->', prices)

### Your turn

In [ ]:
prices = [101.2, 99.8, 103.4, 100.0, 105.5, 108.1]

first_three = todo()   # a list of the first three prices
last_one    = todo()   # just the final price, as a number

print(first_three, last_one)
check('1.3', first_three, last_one)

## 4. Dictionaries, tuples, sets

A **dictionary** stores `key: value` pairs and looks things up by name instead of
by position. When you catch yourself writing "the sector is the third item in the
list", you want a dict.

In [ ]:
sectors = {'ACME': 'Technology', 'CRUX': 'Energy', 'EVER': 'Consumer'}

print(sectors['CRUX'])              # look up by key
sectors['FLUX'] = 'Consumer'        # add a new pair
print(len(sectors), 'tickers')

print('ACME' in sectors)            # membership test — fast, and very common
print(list(sectors.keys()))
print(list(sectors.items()))

In [ ]:
# Asking for a key that is not there is an error. Sometimes that is what you
# want (fail loudly). When it is not, .get() returns a default instead.
try:
    sectors['ZZZZ']
except KeyError as exc:
    print('KeyError:', exc)

print(sectors.get('WXYZ', 'not in our universe'))

Two more containers you should recognise:

```python
point   = (48.7, 2.3)      # tuple — fixed record, cannot be changed afterwards
tickers = {'ACME', 'BOLT'} # set   — unordered, no duplicates, fast membership
```

**Which container when**

| Container | Written as   | Reach for it when |
| --------- | ------------ | ----------------- |
| `list`    | `[1, 2, 3]`  | order matters and you will loop over it |
| `dict`    | `{'a': 1}`   | you look things up by name |
| `tuple`   | `(1, 2)`     | a small fixed record that must not change |
| `set`     | `{1, 2}`     | you only care about membership, or removing duplicates |

Picking the right one is most of what makes code fast *and* readable. Searching a
list of 10,000 names takes 10,000 comparisons; the same lookup in a dict or set
takes one.

### Your turn

In [ ]:
sectors = {'ACME': 'Technology', 'CRUX': 'Energy', 'EVER': 'Consumer'}

crux_sector    = todo()   # the sector CRUX belongs to
missing_sector = todo()   # look up 'ZZZZ' but return 'Unknown' instead of crashing

print(crux_sector, '|', missing_sector)
check('1.4', crux_sector, missing_sector)

## 5. Making decisions: `if` / `elif` / `else`

Python uses **indentation** to show what belongs inside the `if`. Four spaces,
consistently. There are no braces and no `end` keyword — the indentation *is* the
structure, which is why a stray space is a syntax error rather than a style
complaint.

In [ ]:
pnl = -320.0

if pnl > 0:
    label = 'gain'
elif pnl < 0:
    label = 'loss'
else:
    label = 'flat'

print(f'{pnl:+,.2f} is a {label}')

In [ ]:
# Conditions are just expressions that end up True or False.
print(3 > 2, 3 == 3, 3 != 4, 3 >= 4)
print(0 < 5 < 10)                      # chaining works and reads like maths

# and / or / not — spelled out as words, not && || !
is_open, has_cash = True, False
print(is_open and has_cash)
print(is_open or has_cash)
print(not has_cash)

# == compares values. Use `is` only for None / True / False.
print([1, 2] == [1, 2], [1, 2] is [1, 2])

## 6. Loops

A `for` loop walks through a collection one item at a time. You almost never need
to manage an index yourself.

In [ ]:
for p in [101.2, 99.8, 103.4]:
    print(p)

print()
for i in range(3):            # range(3) is 0, 1, 2 — up to but not including 3
    print(i, i ** 2)

In [ ]:
prices  = [101.2, 99.8, 103.4, 100.0, 105.5, 108.1]
tickers = ['ACME', 'BOLT', 'CRUX']

# enumerate: when you need the position as well as the item
for i, p in enumerate(prices[:3]):
    print(i, p)

print()
# zip: walk two collections side by side
for t, p in zip(tickers, prices):
    print(f'{t} {p}')

The **accumulator pattern** is the single most common loop you will write: set up
an empty total (or list) *before* the loop, and add to it *inside*.

```python
total = 0.0              # before
for p in prices:
    total = total + p    # inside   (total += p is shorthand for the same thing)
```

Python has `sum()`, `max()`, `min()`, `len()` built in, and NumPy does it faster
still — but the pattern generalises to things no built-in covers, so it is worth
having in your fingers.

### Your turn

In [ ]:
prices = [101.2, 99.8, 103.4, 100.0, 105.5, 108.1]

# Add up the prices with a loop. Do not use sum() — this one is about the pattern.
total = todo()

print(total)
check('1.5', total)

### Your turn again — a loop with a decision inside it

In [ ]:
returns = [0.012, -0.004, 0.000, 0.008, -0.011, 0.003, -0.002, 0.006]

# Count how many days were up, down, and exactly flat.
up, down, flat = 0, 0, 0

# your loop here

print(f'up {up}, down {down}, flat {flat}')
check('1.6', up, down, flat)

### `while`, and when not to use it

`while` repeats until a condition stops being true. Use it when you genuinely do
not know how many rounds you need — a simulation running until a bankroll hits
zero, say (you will do exactly that in notebook 4).

Do **not** use it to walk through a collection. `while i < len(prices)` with a
manual `i = i + 1` is three lines where `for p in prices` is one, and if you ever
forget the increment the cell hangs forever.

In [ ]:
bankroll, rounds = 10, 0
while bankroll > 0 and rounds < 100:      # always give yourself a second exit
    bankroll -= 3
    rounds += 1

print(f'broke after {rounds} rounds, bankroll {bankroll}')

## 7. Comprehensions

A comprehension is an accumulator loop that builds a list, written on one line:

```python
[ expression   for item in collection   if condition ]
```

Read it left to right as: *"give me this, for every one of those, where that."*

In [ ]:
prices = [101.2, 99.8, 103.4, 100.0, 105.5, 108.1]

# The long way
doubled = []
for p in prices:
    doubled.append(p * 2)

# The same thing
doubled = [p * 2 for p in prices]
print(doubled)

# With a filter: only the items where the condition holds get through
print([p for p in prices if p < 101])

# Comprehensions build dicts too
print({t: len(t) for t in ['ACME', 'BOLT', 'CRUX']})

**When not to use one.** A comprehension should read like a sentence. Stop and
write the plain loop when:

* the body does something rather than producing something (printing, writing a
  file, updating a database) — that is what a `for` loop is for;
* you need two or more `if`s, or a nested loop, and the line no longer fits;
* you had to think for more than a moment to work out what it returns.

Clever is not the goal. Obvious is the goal.

### Your turn

In [ ]:
prices = [101.2, 99.8, 103.4, 100.0, 105.5, 108.1]

# Use a comprehension to build a list of only the prices above 104.
above_104 = todo()

print(above_104)
check('1.7', above_104)

## 8. Functions

A function is a named piece of code you can run again with different inputs.

```python
def name_of_function(argument, another):
    """One line saying what it does."""
    ...
    return something
```

Three rules that matter more than the syntax:

1. **`return` hands a value back. `print` only draws on the screen.** A function
   that prints but does not return gives you `None`, and `None` breaks the next
   calculation. This is the most common beginner bug there is.
2. **Take what you need as arguments; give back what you produced.** A function
   that quietly reads a variable from outside itself works exactly once, in the
   notebook where you wrote it.
3. **Write one as soon as you are tempted to copy-paste.** The second time you
   need something is the moment to name it.

In [ ]:
def position_value(shares, price):
    """Market value of a holding."""
    return shares * price


print(position_value(250, 142.5))
print(position_value(shares=250, price=142.5))   # naming the arguments is clearer
print(position_value(price=142.5, shares=250))   # ...and then order stops mattering

In [ ]:
def annualise(mean_return, periods_per_year=252):
    """Scale a per-period mean return up to a yearly figure.

    periods_per_year has a default, so you can leave it out for daily data.
    """
    return mean_return * periods_per_year


print(annualise(0.0004))              # uses the default 252
print(annualise(0.008, 12))           # monthly data
print(annualise(0.008, periods_per_year=12))

In [ ]:
# Rule 1, demonstrated. This one looks like it works:
def bad_value(shares, price):
    print(shares * price)

result = bad_value(250, 142.5)
print('what came back:', result)   # None — the number went to the screen and vanished
print('so this breaks:')
try:
    print(result * 2)
except TypeError as exc:
    print('   TypeError:', exc)

**Scope.** Names created inside a function live and die there. That is a feature:
it means a function cannot accidentally clobber your variables.

In [ ]:
scale = 100          # outside

def rescale(x):
    scale = 1000     # a different, local `scale`
    return x * scale

print(rescale(2), 'and outside scale is still', scale)

### Your turn

In [ ]:
# Write simple_return(p0, p1) so that it RETURNS the simple return.
# Do not print it. Give it a one-line docstring while you are there.

def simple_return(p0, p1):
    pass    # replace this whole line


print(simple_return(100.0, 110.0))     # should show 0.1
check('1.8', simple_return)            # note: no parentheses — pass the function itself

## 9. Reading errors

Errors are not failure, they are Python telling you precisely what it could not
do. The rule is: **read the last line first.** It has the error type and the
message. The lines above it are the trail of calls that got you there, oldest at
the top — your own code is usually near the bottom.

Here are the ones you will actually meet. (Each is wrapped in `try` so the
notebook keeps running.)

In [ ]:
def show(label, fn):
    try:
        fn()
    except Exception as exc:
        print(f'{label:<14}{type(exc).__name__}: {exc}')


show('typo',      lambda: undefined_name)          # noqa: F821
show('wrong type', lambda: 'abc' + 5)
show('bad key',   lambda: {'a': 1}['b'])
show('bad index', lambda: [1, 2, 3][7])
show('bad value', lambda: int('not a number'))
show('no method', lambda: (3.5).upper())
show('div by 0',  lambda: 1 / 0)

| Error | What it usually means |
| ----- | --------------------- |
| `NameError` | a typo, or you never ran the cell that defines it |
| `TypeError` | the types do not fit — often a `str` where a number belongs, or a `None` that came back from a function that printed instead of returning |
| `KeyError` | that key is not in the dict (check spelling and case) |
| `IndexError` | the list is shorter than you think |
| `ValueError` | right type, impossible value |
| `AttributeError` | that object has no such method — usually you have a different type than you assumed |
| `ModuleNotFoundError` | the package is not installed *in the interpreter you are running* |
| `IndentationError` | mixed tabs and spaces, or a line at the wrong depth |

**On `try` / `except`:** use it only where you can actually do something about the
failure — a download that may time out, a file that may not exist. Wrapping code
in `try: ... except: pass` to make an error message go away does not fix the bug;
it hides it until it costs you far more.

## 10. NumPy — arrays and vectorisation

A NumPy array is a block of numbers, all the same type, laid out together in
memory. That single design decision is why arithmetic on arrays happens **all at
once** instead of one item at a time.

In [ ]:
import numpy as np

prices_list = [101.2, 99.8, 103.4, 100.0, 105.5, 108.1]
prices = np.array(prices_list)

print(prices)
print('shape', prices.shape, '| dtype', prices.dtype)

print(prices * 2)          # every element, no loop
print(prices - 100)
print(prices > 103)        # a comparison gives an array of True/False

In [ ]:
# A list does NOT behave like this — worth seeing once so it never surprises you.
print(prices_list * 2)     # repeats the list instead of doubling the numbers

### Why bother: the same job, timed

In [ ]:
import time

big = np.random.default_rng(0).normal(size=2_000_000)

t0 = time.perf_counter()
out_loop = [x * 2 + 1 for x in big]
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
out_vec = big * 2 + 1
t_vec = time.perf_counter() - t0

print(f'loop      {t_loop:.4f}s')
print(f'vectorised {t_vec:.4f}s')
print(f'{t_loop / t_vec:.0f}x faster, and one line instead of a loop')

This is the habit to build: **if you are writing a `for` loop over numbers, there
is usually an array operation that replaces it.** Shorter, faster, and far less
room for an off-by-one mistake.

In [ ]:
# Aggregations
print(prices.mean(), prices.std(), prices.min(), prices.max(), prices.sum())

# Two dimensions: axis=0 goes down the columns, axis=1 goes across the rows
grid = np.array([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0]])
print(grid.shape)
print('down  columns', grid.mean(axis=0))
print('across rows  ', grid.mean(axis=1))

In [ ]:
# Boolean masks: the most useful idea in NumPy
returns = np.array([0.012, -0.004, 0.000, 0.008, -0.011])

mask = returns < 0
print(mask)
print('the negative ones  ', returns[mask])
print('how many           ', mask.sum())      # True counts as 1
print('what share         ', mask.mean())     # ...so the mean is the proportion
print('floored at zero    ', np.where(returns < 0, 0.0, returns))

**Shifting an array against itself** is how you turn a price series into a return
series without a loop. `p[1:]` is everything from the second element on; `p[:-1]`
is everything except the last. Line them up and divide.

In [ ]:
p = np.array([100.0, 110.0, 99.0])
print('p[1:] ', p[1:])
print('p[:-1]', p[:-1])
print('returns', p[1:] / p[:-1] - 1.0)

### Your turn

In [ ]:
prices = np.array([101.2, 99.8, 103.4, 100.0, 105.5, 108.1])

# Daily simple returns, with no loop. Six prices give five returns.
rets = todo()

print(rets)
check('1.9', rets)

In [ ]:
# What share of those returns were negative? One line, using a boolean mask.
share_negative = todo()

print(share_negative)
check('1.10', share_negative)

**When NOT to use a plain NumPy array:** as soon as your data has labels —
column names, dates, tickers — an array makes you remember that "column 3 is
volume". That is what pandas is for.

## 11. pandas — labelled data

Two objects, and everything else follows:

* a **`Series`** is a labelled column — values plus an index;
* a **`DataFrame`** is a table of Series sharing one index.

Think of a DataFrame as a spreadsheet you drive with code.

In [ ]:
import pandas as pd

s = pd.Series([101.2, 99.8, 103.4], index=['Mon', 'Tue', 'Wed'], name='ACME')
print(s)
print()
print(s['Tue'], '|', s.mean())

### Reading a file

`pd.read_csv` is the workhorse. `DATA_DIR` came from the setup cell, so this
works on anyone's machine — never paste an absolute path like
`/Users/you/Desktop/...` into a notebook other people will run. Build the path
from something the code works out for itself.

The data is **simulated**, not real: six fictional companies over three years.
`generate_data.py`, next to this notebook, shows exactly how it was made.

In [ ]:
prices = pd.read_csv(DATA_DIR / 'stock_prices.csv', parse_dates=['date'])

print(prices.shape)      # (rows, columns)
prices.head()

`parse_dates=['date']` matters. Without it the dates arrive as text, and text
sorts `'2021-10-01'` before `'2021-9-01'`. Always parse your dates on the way in.

Three commands to run on **every** new dataset, before anything else:

In [ ]:
prices.info()          # column names, types, and how many non-null values

In [ ]:
prices.describe()      # count, mean, spread and quartiles for numeric columns

In [ ]:
print(prices.isna().sum())                 # missing values per column
print()
print(prices['ticker'].value_counts())     # how many rows per ticker
print()
print(prices['date'].min(), 'to', prices['date'].max())

Seventeen missing volumes — real data is always a bit broken, and finding out on
day one is much cheaper than finding out after a week of analysis.

### Selecting

In [ ]:
print(type(prices['price']))        # one column -> Series
print(type(prices[['ticker', 'price']]))   # a LIST of columns -> DataFrame

# .iloc is by position, .loc is by label. Rows first, then columns.
print(prices.iloc[0])
print()
print(prices.loc[0:2, ['date', 'ticker', 'price']])

In [ ]:
# Filtering: build a boolean Series, then use it to pick rows.
is_acme = prices['ticker'] == 'ACME'
acme = prices[is_acme]
print(acme.shape)

# Combine conditions with & and | — and the brackets around each are required.
big_tech = prices[(prices['sector'] == 'Technology') & (prices['price'] > 200)]
print(big_tech.shape)
big_tech.head(3)

⚠️ **Assign through `.loc`, never through two sets of brackets.**

```python
prices[prices['ticker'] == 'ACME']['price'] = 0     # silently does nothing useful
prices.loc[prices['ticker'] == 'ACME', 'price'] = 0 # correct
```

The first one changes a temporary copy that is thrown away a microsecond later.
This is the single most common pandas bug.

### Grouping — the one that does the real work

In [ ]:
# "Split the rows by ticker, then average the price within each group."
print(prices.groupby('ticker')['price'].mean().round(2))
print()

# Several statistics at once, on several columns
print(prices.groupby('ticker').agg(
    first_price=('price', 'first'),
    last_price=('price', 'last'),
    avg_volume=('volume', 'mean'),
).round(2))

In [ ]:
# Reshaping: long (one row per date-ticker) -> wide (one column per ticker)
wide = prices.pivot(index='date', columns='ticker', values='price')
wide.head()

In [ ]:
# Wide layout makes whole-table maths easy
daily_returns = wide.pct_change().dropna()
print(daily_returns.std().sort_values(ascending=False).round(4))

### Your turn

In [ ]:
# What was ACME's total return over the whole sample?
# Filter to ACME, then compare the last price with the first.
# Use .iloc[0] and .iloc[-1] on the price column.

acme_total_return = todo()

print(f'{acme_total_return:.2%}')
check('1.11', acme_total_return)

In [ ]:
# Average daily volume for each sector, as a pandas Series with the sector
# names as its index.

volume_by_sector = todo()

print(volume_by_sector)
check('1.12', volume_by_sector)

## 12. Classes

A class bundles **data** and the **functions that work on that data** into one
object. `__init__` runs when you create one; `self` is the object itself, and it
is how a method reaches the data stored on it.

In [ ]:
class Account:
    """A cash account you can pay into and out of."""

    def __init__(self, owner, balance=0.0):
        self.owner = owner           # stored on this particular object
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount       # methods can change the object's own data
        return self.balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError(f'{self.owner} only has {self.balance:,.2f}')
        self.balance -= amount
        return self.balance

    def __repr__(self):              # what Python shows when you print it
        return f'Account({self.owner}, {self.balance:,.2f})'


desk = Account('trading desk', 10_000.0)
petty = Account('petty cash')            # balance uses its default of 0.0

desk.deposit(2_500.0)
print(desk, '|', petty)

# Each object keeps its own copy of the data — that is the whole point.
try:
    petty.withdraw(50.0)
except ValueError as exc:
    print('ValueError:', exc)

`Account` bundles two pieces of state (`owner`, `balance`) with the three things
you can do to them. Every `Account` you create carries its own balance, and
`withdraw` can enforce a rule about it — that is state and behaviour in one place.

**When a class earns its keep:** when several functions all need the same few
pieces of data, and you are tired of passing them around together. `DataDefinition`
in this repo is exactly that — it holds a source, an item and a date range, and
its methods all work on those. You met it in the main README:

```python
dd = DataDefinition(source='yfin', item='SPY', start='2000-01-01', end=None)
prices = dd.extract()
```

**When a class is the wrong tool:** when a function would do. If your class has
one method and you construct it, call it, and throw it away, you have written a
function with extra ceremony. Start with functions; promote to a class when the
data starts travelling in a group.

### Your turn

In [ ]:
# Write the Position class yourself.
#   __init__(self, ticker, shares, price)  stores all three on self
#   market_value(self)                     returns shares * price

class Position:
    pass   # replace this line with __init__ and market_value


# Once it works, uncomment these two lines — they should print
#   ACME 250 142.5 35625.0
# p = Position('ACME', 250, 142.5)
# print(p.ticker, p.shares, p.price, p.market_value())

check('1.13', Position)     # the class itself, not an instance

## 13. Notebooks, scripts and imports

A notebook is for exploring: you look at something, change it, look again.

A **`.py` file** is for code you want to reuse. Once something works, move it out
of the notebook and into a module — then every notebook, script, and teammate can
`import` the one copy, and fixing a bug fixes it everywhere.

This repository is laid out that way already:

```python
from Utilities.tools import compute_levels_from_returns, return_descriptor
from Data.data_definition import DataDefinition
```

You can call those from here too, as long as Python can see the project root.

In [ ]:
from workbook import PROJECT_ROOT      # the repo root, worked out from workbook.py

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Utilities.tools import compute_levels_from_returns

growth = compute_levels_from_returns(daily_returns['ACME'])
print(f'$1 in ACME became ${growth.iloc[-1]:.2f}')
print(compute_levels_from_returns.__doc__.splitlines()[0])

**Three import styles, and when to use each**

```python
import numpy as np                      # whole package, short alias — the standard
from Utilities.tools import ffill_na    # one name, when you use it a lot
from Utilities import tools             # the module, then tools.ffill_na(...)
```

Avoid `from numpy import *`. It dumps hundreds of names into your session and you
will never work out which one shadowed your variable.

**`if __name__ == '__main__':`** at the bottom of a `.py` file means "only run
this bit when the file is executed directly, not when it is imported". That is
why `Utilities/tools.py` can hold a self-test at the end without that test firing
every time you import the module.

## 14. Where you are

You can now read and write the constructs that make up nearly all day-to-day
analysis code:

| | |
| --- | --- |
| **values** | `str`, `int`, `float`, `bool`, and why the difference matters |
| **containers** | list, dict, tuple, set — and which one to reach for |
| **control** | `if` / `elif` / `else`, `for`, `while`, comprehensions |
| **structure** | functions, classes, modules, imports |
| **numbers** | NumPy arrays, vectorisation, boolean masks |
| **tables** | pandas: read, inspect, filter, group, reshape |
| **failure** | reading a traceback, and where `try` belongs |

### Check your work

In [ ]:
from workbook import exercises
print('Notebook 1 exercises:', ', '.join(exercises(1)))
print()
print('Re-run any cell above to re-check it, or call hint(key) for a nudge:')
hint('1.12')

### Next

**Notebook 2 — Plotting.** You have a table of prices; now make a figure of it
that someone else would be happy to look at. Same data, one new skill.